<a href="https://colab.research.google.com/github/Soham12Sawant/23102C0036_R_Programming/blob/main/R_Lab_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install and load packages
packages <- c("readxl", "dplyr", "tidyr", "ggplot2", "cluster",
              "factoextra", "randomForest", "e1071", "pROC",
              "caret", "plotly", "scales")
installed <- rownames(installed.packages())
for (p in packages) {
  if (!(p %in% installed)) install.packages(p, repos="https://cloud.r-project.org")
}

library(readxl); library(dplyr); library(tidyr); library(ggplot2)
library(cluster); library(factoextra); library(randomForest)
library(e1071); library(pROC); library(caret); library(plotly); library(scales)
set.seed(123)


In [ ]:
dataset_url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
local_file <- "Online Retail.xlsx"
if (!file.exists(local_file)) download.file(dataset_url, local_file, mode="wb")
retail <- read_excel(local_file)
cat("Rows:", nrow(retail), " Columns:", ncol(retail), "\n")
head(retail)


In [ ]:
# Inspect data types and missing values
str(retail)
missing_summary <- data.frame(
  Column=names(retail),
  Missing=sapply(retail, function(x) sum(is.na(x))),
  Missing_Percent=round(sapply(retail, function(x) mean(is.na(x))*100),2)
)
missing_summary


In [ ]:
original_rows <- nrow(retail)
retail <- retail %>%
  mutate(InvoiceDate=as.POSIXct(InvoiceDate),
         CustomerID=as.character(CustomerID),
         InvoiceNo=as.character(InvoiceNo))

retail_clean <- retail %>%
  filter(!grepl("^C", InvoiceNo), !is.na(CustomerID),
         Quantity > 0, UnitPrice > 0) %>%
  mutate(TotalPrice=Quantity*UnitPrice)

cat("Original rows:", original_rows, "\n")
cat("Clean rows:", nrow(retail_clean), "\n")
cat("Removed:", original_rows-nrow(retail_clean), "\n")


In [ ]:
# Transaction-level distributions
ggplot(retail_clean, aes(Quantity)) +
  geom_histogram(bins=60) + scale_x_continuous(trans="log1p") +
  labs(title="Quantity Distribution", x="Quantity (log1p)", y="Count") + theme_minimal()

ggplot(retail_clean, aes(UnitPrice)) +
  geom_histogram(bins=60) + scale_x_continuous(trans="log1p") +
  labs(title="Unit Price Distribution", x="Unit Price (log1p)", y="Count") + theme_minimal()


In [ ]:
reference_date <- max(retail_clean$InvoiceDate) + 1
customer_features <- retail_clean %>%
  group_by(CustomerID) %>%
  summarise(
    Recency=as.numeric(difftime(reference_date,max(InvoiceDate),units="days")),
    Frequency=n_distinct(InvoiceNo),
    Monetary=sum(TotalPrice),
    AvgTransactionValue=Monetary/Frequency,
    QuantityPurchased=sum(Quantity),
    ActiveMonths=n_distinct(format(InvoiceDate,"%Y-%m")),
    .groups="drop"
  ) %>%
  mutate(PurchaseFrequency=Frequency/ActiveMonths)

dim(customer_features)
head(customer_features)


In [ ]:
cap_outliers <- function(x, lower=0.01, upper=0.99) {
  q <- quantile(x, c(lower,upper), na.rm=TRUE)
  pmin(pmax(x,q[1]),q[2])
}

feature_names <- c("Recency","Frequency","Monetary","AvgTransactionValue",
                   "QuantityPurchased","PurchaseFrequency")
customer_capped <- customer_features
for (f in feature_names) customer_capped[[f]] <- cap_outliers(customer_capped[[f]])

customer_model <- customer_capped %>%
  mutate(Recency_log=log1p(Recency), Frequency_log=log1p(Frequency),
         Monetary_log=log1p(Monetary), AvgTransactionValue_log=log1p(AvgTransactionValue),
         QuantityPurchased_log=log1p(QuantityPurchased),
         PurchaseFrequency_log=log1p(PurchaseFrequency))

model_features <- c("Recency_log","Frequency_log","Monetary_log",
                    "AvgTransactionValue_log","QuantityPurchased_log","PurchaseFrequency_log")
scaled_df <- as.data.frame(scale(customer_model[,model_features]))
colnames(scaled_df) <- feature_names
summary(scaled_df)


In [ ]:
k_values <- 2:10
wss <- sapply(k_values, function(k) kmeans(scaled_df, centers=k, nstart=25)$tot.withinss)
elbow_data <- data.frame(K=k_values, WCSS=wss)

ggplot(elbow_data,aes(K,WCSS)) + geom_line() + geom_point(size=3) +
  labs(title="Elbow Method",x="Number of Clusters (K)",y="Within-Cluster Sum of Squares") +
  theme_minimal()

# Elbow heuristic
x <- elbow_data$K; y <- elbow_data$WCSS
line_y <- y[1] + (y[length(y)]-y[1])*(x-x[1])/(x[length(x)]-x[1])
distance <- abs(y-line_y)/sqrt(1+((y[length(y)]-y[1])/(x[length(x)]-x[1]))^2)
K <- x[which.max(distance)]
cat("Suggested K:",K,"\n")


In [ ]:
kmeans_result <- kmeans(scaled_df, centers=K, nstart=50, iter.max=100)
customer_segmented <- customer_model %>% mutate(Cluster=factor(kmeans_result$cluster))
cluster_sizes <- customer_segmented %>% count(Cluster,name="Customers") %>%
  mutate(Percentage=round(Customers/sum(Customers)*100,2))
cluster_sizes


In [ ]:
silhouette_kmeans <- silhouette(kmeans_result$cluster, dist(scaled_df))
avg_silhouette <- mean(silhouette_kmeans[,"sil_width"])
cat("K-Means Silhouette Score:",round(avg_silhouette,4),"\n")
plot(silhouette_kmeans, main=paste("K-Means Silhouette - K =",K))


In [ ]:
distance_matrix <- dist(scaled_df, method="euclidean")
hc_result <- hclust(distance_matrix, method="ward.D2")
plot(hc_result, labels=FALSE, hang=-1, main="Hierarchical Clustering Dendrogram",
     xlab="Customers", ylab="Height")
rect.hclust(hc_result,k=K)
hier_cluster <- cutree(hc_result,k=K)
hier_silhouette <- silhouette(hier_cluster,distance_matrix)
avg_hier_silhouette <- mean(hier_silhouette[,"sil_width"])
cat("Hierarchical Silhouette Score:",round(avg_hier_silhouette,4),"\n")


In [ ]:
clustering_comparison <- data.frame(
  Method=c("K-Means","Hierarchical"),
  Silhouette_Score=c(avg_silhouette,avg_hier_silhouette)
)
clustering_comparison


In [ ]:
pca_result <- prcomp(scaled_df, center=FALSE, scale.=FALSE)
variance_explained <- pca_result$sdev^2/sum(pca_result$sdev^2)
pca_variance <- data.frame(Component=paste0("PC",seq_along(variance_explained)),
                            Variance=variance_explained,Cumulative=cumsum(variance_explained))
pca_variance


In [ ]:
ggplot(pca_variance,aes(Component,Variance)) + geom_col() +
  labs(title="PCA Explained Variance",x="Principal Component",y="Variance Explained") + theme_minimal()

pca_customers <- data.frame(PC1=pca_result$x[,1],PC2=pca_result$x[,2],PC3=pca_result$x[,3],
                            Cluster=customer_segmented$Cluster)

ggplot(pca_customers,aes(PC1,PC2,shape=Cluster)) + geom_point(alpha=.65) +
  labs(title="Customer Segments – 2D PCA Projection",x="PC1",y="PC2") + theme_minimal()


In [ ]:
cluster_profile <- customer_segmented %>% group_by(Cluster) %>% summarise(
  Customers=n(), Recency=mean(Recency), Frequency=mean(Frequency),
  Monetary=mean(Monetary), AvgTransactionValue=mean(AvgTransactionValue),
  QuantityPurchased=mean(QuantityPurchased), PurchaseFrequency=mean(PurchaseFrequency),
  .groups="drop") %>% mutate(Share=round(Customers/sum(Customers)*100,2))
cluster_profile


In [ ]:
profile_long <- cluster_profile %>%
  select(Cluster,Recency,Frequency,Monetary,AvgTransactionValue,QuantityPurchased,PurchaseFrequency) %>%
  pivot_longer(-Cluster,names_to="Feature",values_to="Value") %>%
  group_by(Feature) %>% mutate(RelativeValue=as.numeric(scale(Value))) %>% ungroup()

ggplot(profile_long,aes(Feature,Cluster,fill=RelativeValue)) + geom_tile() +
  geom_text(aes(label=round(RelativeValue,2)),size=3) +
  labs(title="Relative Cluster Profile",x="Feature",y="Cluster",fill="Relative Level") +
  theme_minimal() + theme(axis.text.x=element_text(angle=45,hjust=1))


In [ ]:
high_value_cluster <- cluster_profile$Cluster[which.max(cluster_profile$Monetary)]
cat("High-value cluster:",as.character(high_value_cluster),"\n")

customer_model <- customer_segmented %>%
  mutate(HighValue=factor(ifelse(Cluster==high_value_cluster,"Yes","No"),levels=c("No","Yes")))
table(customer_model$HighValue)


In [ ]:
predictor_names <- feature_names
model_data <- customer_model %>% select(all_of(predictor_names),HighValue) %>% drop_na()
set.seed(123)
train_index <- createDataPartition(model_data$HighValue,p=.80,list=FALSE)
train_data <- model_data[train_index,]; test_data <- model_data[-train_index,]
cat("Training rows:",nrow(train_data)," Testing rows:",nrow(test_data),"\n")
prop.table(table(train_data$HighValue)); prop.table(table(test_data$HighValue))


In [ ]:
set.seed(123)
rf_model <- randomForest(HighValue~.,data=train_data,ntree=300,importance=TRUE)
rf_pred <- predict(rf_model,test_data,type="response")
rf_prob <- predict(rf_model,test_data,type="prob")[,"Yes"]
rf_model


In [ ]:
set.seed(123)
svm_model <- svm(HighValue~.,data=train_data,kernel="radial",probability=TRUE,scale=TRUE)
svm_pred <- predict(svm_model,test_data,probability=TRUE)
svm_prob <- attr(svm_pred,"probabilities")[,"Yes"]
svm_pred


In [ ]:
calculate_metrics <- function(actual,predicted,probability) {
  actual <- factor(actual,levels=c("No","Yes")); predicted <- factor(predicted,levels=c("No","Yes"))
  cm <- confusionMatrix(predicted,actual,positive="Yes")
  roc_obj <- roc(actual,probability,levels=c("No","Yes"),direction="<")
  data.frame(Accuracy=cm$overall["Accuracy"],Precision=cm$byClass["Pos Pred Value"],
             Recall=cm$byClass["Sensitivity"],F1_Score=cm$byClass["F1"],
             ROC_AUC=as.numeric(auc(roc_obj)))
}

rf_metrics <- calculate_metrics(test_data$HighValue,rf_pred,rf_prob)
svm_metrics <- calculate_metrics(test_data$HighValue,svm_pred,svm_prob)
model_comparison <- rbind(Random_Forest=rf_metrics,SVM=svm_metrics)
round(model_comparison,4)

cat("\nRandom Forest Confusion Matrix\n"); rf_cm <- confusionMatrix(rf_pred,test_data$HighValue,positive="Yes"); rf_cm
cat("\nSVM Confusion Matrix\n"); svm_cm <- confusionMatrix(svm_pred,test_data$HighValue,positive="Yes"); svm_cm


In [ ]:
rf_roc <- roc(test_data$HighValue,rf_prob,levels=c("No","Yes"),direction="<")
svm_roc <- roc(test_data$HighValue,svm_prob,levels=c("No","Yes"),direction="<")
plot(rf_roc,main="ROC Curve – Random Forest vs SVM",lwd=2)
lines(svm_roc,lwd=2,lty=2)
legend("bottomright",legend=c(paste0("Random Forest AUC = ",round(auc(rf_roc),3)),
                               paste0("SVM AUC = ",round(auc(svm_roc),3))),lwd=2,lty=c(1,2))


In [ ]:
importance_values <- importance(rf_model)
importance_df <- data.frame(Feature=rownames(importance_values),
  MeanDecreaseAccuracy=importance_values[,"MeanDecreaseAccuracy"],
  MeanDecreaseGini=importance_values[,"MeanDecreaseGini"]) %>%
  arrange(desc(MeanDecreaseAccuracy))
importance_df
varImpPlot(rf_model,main="Random Forest Feature Importance")


In [ ]:
pca_3d <- data.frame(PC1=pca_result$x[,1],PC2=pca_result$x[,2],PC3=pca_result$x[,3],
                     Cluster=customer_segmented$Cluster,CustomerID=customer_segmented$CustomerID)
plot_ly(pca_3d,x=~PC1,y=~PC2,z=~PC3,type="scatter3d",mode="markers",color=~Cluster,
        text=~paste("Customer:",CustomerID,"<br>Cluster:",Cluster),hoverinfo="text") %>%
  layout(title="Interactive 3D Customer Segmentation",
         scene=list(xaxis=list(title="PC1"),yaxis=list(title="PC2"),zaxis=list(title="PC3")))


In [ ]:
marketing_recommendations <- cluster_profile %>% rowwise() %>% mutate(
  Strategy=case_when(
    Cluster==high_value_cluster ~ "VIP rewards, loyalty benefits, personalized offers and premium cross-selling.",
    Recency>median(cluster_profile$Recency) & Frequency<median(cluster_profile$Frequency) ~
      "Reactivation campaigns, reminders and limited-time discounts.",
    Frequency>=median(cluster_profile$Frequency) & Monetary<median(cluster_profile$Monetary) ~
      "Upselling, bundles and cross-selling to increase basket value.",
    TRUE ~ "Personalized recommendations, engagement campaigns and loyalty incentives."
  )) %>% ungroup() %>%
  select(Cluster,Customers,Share,Monetary,Frequency,Recency,Strategy)
marketing_recommendations


In [ ]:
cat("========== EXPERIMENT SUMMARY ==========\n")
cat("Customers:",nrow(customer_segmented),"\n")
cat("Selected K:",K,"\n")
cat("K-Means Silhouette:",round(avg_silhouette,4),"\n")
cat("Hierarchical Silhouette:",round(avg_hier_silhouette,4),"\n")
cat("High-value cluster:",as.character(high_value_cluster),"\n\n")
print(round(model_comparison,4))
cat("\nBest model by ROC-AUC:",rownames(model_comparison)[which.max(model_comparison$ROC_AUC)],"\n")
